# fasthtml

> In-kernel live inspector panel served via FastHTML

In [ ]:
#| default_exp fasthtml

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
import json
from urllib.parse import quote
from fasthtml.common import to_xml
from starlette.testclient import TestClient
import paar.bridge as B, paar.fasthtml as F
from paar.fasthtml import app, _node, _broadcast, _acc
from paar.core import VarInfo

class _IP:
    def __init__(self, ns): self.user_ns=ns; self.user_ns_hidden=set()
    class events:
        @staticmethod
        def register(n, fn): pass

def test_node_scalar_no_toggle():
    html = to_xml(_node(VarInfo(name='n', type='int', value='5', accessor=('n',), path='n')))
    assert 'n' in html and 'int' in html and '5' in html
    assert '/expand' not in html and '<details' not in html
def test_node_container_details():
    html = to_xml(_node(VarInfo(name='d', type='dict', value='{...}',
                                is_container=True, accessor=('d',), path='d')))
    assert '<details' in html and '<summary' in html
    assert 'paar-children' in html and '/expand?accessor=' in html
def test_node_container_lazy_once():
    html = to_xml(_node(VarInfo(name='d', type='dict', is_container=True, accessor=('d',), path='d')))
    assert 'once' in html   # hx-trigger="click once" -> children fetched only on first open
def test_rows_route_lists_vars():
    B.get_ipython = lambda: _IP({'alpha': 123})
    r = TestClient(app).get('/rows')
    assert r.status_code==200 and 'alpha' in r.text and '123' in r.text and 'id="rows"' in r.text
def test_home_wires_ws_and_loader():
    r = TestClient(app).get('/')
    assert r.status_code==200
    assert 'ws-connect="/live"' in r.text and 'hx-ext="ws"' in r.text
    assert 'id="rows"' in r.text and '/rows' in r.text
def test_home_forces_dark_theme():
    r = TestClient(app).get('/')
    assert r.status_code==200 and 'data-theme="dark"' in r.text
def test_expand_route_returns_children():
    B.get_ipython = lambda: _IP({'d': {'x': 1}})
    r = TestClient(app).get('/expand?accessor=' + quote(json.dumps(['d'])))
    assert r.status_code==200 and 'x' in r.text and '1' in r.text
def test_broadcast_drops_dead_clients():
    F._clients.clear()
    def _bad_send(_): raise RuntimeError('dead')
    F._clients.append(('not-a-loop', _bad_send))
    _broadcast('x')
    assert F._clients==[]
def test_node_grid_has_grid_link():
    html = to_xml(_node(VarInfo(name='arr', type='ndarray', is_grid=True, accessor=('arr',), path='arr')))
    assert '<details' in html and '/grid?accessor=' in html and 'grid' in html
    assert '/expand' not in html   # gridables are not tree-expanded
def test_grid_route_renders_table():
    import numpy as np
    B.get_ipython = lambda: _IP({'arr': np.arange(6).reshape(2,3)})
    r = TestClient(app).get('/grid?accessor=' + quote(json.dumps(['arr'])) + '&roff=0&coff=0')
    assert r.status_code==200 and '<table' in r.text and '5' in r.text  # last cell value present
for t in [test_node_scalar_no_toggle,test_node_container_details,test_node_container_lazy_once,
          test_rows_route_lists_vars,test_home_wires_ws_and_loader,test_home_forces_dark_theme,
          test_expand_route_returns_children,test_broadcast_drops_dead_clients,
          test_node_grid_has_grid_link,test_grid_route_renders_table]: t()

In [ ]:
#| export
import asyncio, threading, json
from urllib.parse import quote
from fasthtml.common import *
from fasthtml.jupyter import JupyUvi, HTMX
from paar.bridge import Bridge, on_change
from paar.core import VarInfo

_css = Style(
    '.paar-children{margin-left:1rem} '
    'summary{cursor:pointer} '
    '.paar-node{font-size:.9rem} '
    'code{white-space:pre} '
    '.paar-grid{max-height:360px;overflow:auto} '
    '.paar-grid table{font-size:.8rem;margin:0} '
    '.paar-grid th{position:sticky;top:0;background:var(--pico-background-color)} '
    '.paar-gridnav{margin:.25rem 0} '
    '.paar-page{margin-right:.5rem;cursor:pointer} '
    '.paar-grid-label{opacity:.6}')
bridge = Bridge()
app = FastHTML(exts='ws', hdrs=(_css,), htmlkw={'data-theme':'dark'})   # ws ext + pico dark theme (readable in dark IDEs)
_clients = []   # list[(loop, send)]
_clients_lock = threading.Lock()
_server = None

In [ ]:
#| export
def _badges(v:VarInfo):
    "shape/dtype summary string for a row."
    return ' '.join(x for x in [v.shape and f'shape={v.shape}',
                                v.dtype and f'dtype={v.dtype}'] if x)

def _acc(accessor):
    "URL-safe encoding of a positional accessor."
    return quote(json.dumps(list(accessor)), safe='')

def _node(v:VarInfo):
    "Render a tree node; containers expand, gridables open a paged grid, scalars are plain."
    b = _badges(v)
    head = Span(Strong(v.name), ' ', Small(v.type, title=v.qualifier), ' ',
                Code(v.value), (' ' + b) if b else '',
                cls=('error' if v.is_error else None))
    if v.is_grid:
        return Details(
            Summary(head, ' ', Span('▦ grid', cls='paar-grid-label'),
                    hx_get=f'/grid?accessor={_acc(v.accessor)}&roff=0&coff=0',
                    hx_target='next .paar-children', hx_swap='innerHTML', hx_trigger='click once'),
            Div(cls='paar-children'), cls='paar-node')
    if v.is_container:
        return Details(
            Summary(head, hx_get=f'/expand?accessor={_acc(v.accessor)}',
                    hx_target='next .paar-children', hx_swap='innerHTML', hx_trigger='click once'),
            Div(cls='paar-children'), cls='paar-node')
    return Div(head, cls='paar-node')

def _grid_nav(page, acc):
    "Prev/next paging controls + window info for a grid page."
    roff, coff = page['roff'], page['coff']
    npr, npc = len(page['index']), len(page['headers'])
    nrows, ncols = page['nrows'], page['ncols']
    pr, pc = page['rows'], page['cols']
    def lnk(label, ro, co):
        return A(label, hx_get=f'/grid?accessor={_acc(acc)}&roff={ro}&coff={co}',
                 hx_target='closest .paar-children', hx_swap='innerHTML', cls='paar-page')
    ctrls = []
    if roff > 0: ctrls.append(lnk('◀ rows', max(0, roff-pr), coff))
    if roff+npr < nrows: ctrls.append(lnk('rows ▶', roff+npr, coff))
    if coff > 0: ctrls.append(lnk('◀ cols', roff, max(0, coff-pc)))
    if coff+npc < ncols: ctrls.append(lnk('cols ▶', roff, coff+npc))
    info = Small(f'rows {roff}-{roff+npr} of {nrows}, cols {coff}-{coff+npc} of {ncols}')
    return Div(info, ' ', *ctrls, cls='paar-gridnav')

def _grid(page, acc):
    "Render a grid page as a scrollable table with paging controls."
    if page is None: return Div('not gridable')
    hdr = Tr(Th(''), *[Th(h) for h in page['headers']])
    body = [Tr(Th(ix), *[Td(c) for c in row]) for ix, row in zip(page['index'], page['cells'])]
    return Div(Div(Table(Thead(hdr), Tbody(*body)), cls='paar-grid'),
               _grid_nav(page, acc))

def _rows_div():
    "The rendered snapshot as a node tree, wrapped in the #rows target."
    return Div(*[_node(v) for v in bridge.snapshot()], id='rows')

def _loader(oob=False):
    "A #rows div that immediately GETs /rows (used initially and as the WS nudge)."
    kw = dict(hx_get='/rows', hx_trigger='load', hx_swap='outerHTML')
    if oob: kw['hx_swap_oob']='true'
    return Div(id='rows', **kw)

In [ ]:
#| export
@app.get('/')
def home():
    return Titled('paar', Div(_loader(), hx_ext='ws', ws_connect='/live', id='paar'))

@app.get('/rows')
def rows(): return _rows_div()

@app.get('/expand')
def expand_route(accessor:str):
    return Div(*[_node(v) for v in bridge.expand(tuple(json.loads(accessor)))])

@app.get('/grid')
def grid_route(accessor:str, roff:int=0, coff:int=0):
    acc = tuple(json.loads(accessor))
    return _grid(bridge.grid(acc, roff, coff), acc)

def _drop(send):
    "Remove a client by its send identity (thread-safe)."
    with _clients_lock:
        _clients[:] = [(l,s) for (l,s) in _clients if s is not send]

async def _conn(send):
    with _clients_lock: _clients.append((asyncio.get_running_loop(), send))
async def _disconn(send): _drop(send)

@app.ws('/live', conn=_conn, disconn=_disconn)
async def live(send): pass

In [ ]:
#| export
def _broadcast(fragment):
    "Push fragment to every WS client from any thread; drop clients that fail."
    with _clients_lock: targets = list(_clients)
    for loop, send in targets:
        try: fut = asyncio.run_coroutine_threadsafe(send(fragment), loop)
        except Exception: _drop(send); continue
        fut.add_done_callback(lambda f, s=send: (not f.cancelled()) and f.exception() is not None and _drop(s))

def inspector(port=8000, height=520):
    "Start the in-kernel live inspector panel and return the inline iframe."
    global _server
    if _server is None:
        _server = JupyUvi(app, port=port, daemon=True)
        on_change(lambda: _broadcast(to_xml(_loader(oob=True))))
    return HTMX(port=port, height=height, link=True)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()